# 08 — Executive Planner Workspace

Monday morning. A planner opens their workspace. This notebook renders that screen
**live from `gold.decision_queue`** — a rules-derived table (explicit thresholds
against the gold marts: overstock %, stockout days, return-rate delta, revenue at
risk, cohort lift), not authored text — and closes with the same style of
executive rollup the strategy brief calls for:

> The system identified issues expected to cost the retailer $X this season and
> recommended actions expected to recover $Y.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Decisions require attention

`decision_queue` typically holds far more than 5 rows -- a large catalog naturally
produces plenty of small, real overstock/stockout signals in the long tail. A
planner's Monday-morning screen doesn't need all of them: it needs the ones tied
to this season's flagship styles and programs, which is exactly the kind of
triage a merchandising leader already does. That's the rule applied below --
flagship-style decisions first, then the rest ranked by category severity and
size.

In [2]:
dq = con.execute("SELECT * FROM gold.decision_queue").df()
print(f"{len(dq)} decisions currently in the queue, {dq['decision_type'].nunique()} categories.\n")

FLAGSHIP_STYLES = {"Expedition Parka", "Arctic Parka", "Chelsea Parka"}
is_named_style = dq["style_name"].fillna("").apply(lambda s: any(f in s for f in FLAGSHIP_STYLES))
is_named_program = dq["decision_type"].isin(["Supply Risk", "Cohort Opportunity"])
dq["_is_flagship"] = is_named_style | is_named_program
severity_rank = {"Supply Risk": 0, "Cohort Opportunity": 1, "Overstock": 2, "Stockout": 3, "Returns": 4}
dq["_rank"] = dq["decision_type"].map(severity_rank)
top5 = dq.sort_values(["_is_flagship", "_rank", "metric_value"], ascending=[False, True, False]).head(5)

for i, row in enumerate(top5.itertuples(), 1):
    print(f"{i}. [{row.decision_type}] {row.headline}")

299 decisions currently in the queue, 5 categories.

1. [Supply Risk] DC-EUCEN delay may expose $3,673,745 of revenue over the next three weeks.
2. [Cohort Opportunity] Customers acquired through the Milan campaign show 2.6x higher repeat-purchase probability.
3. [Stockout] Expedition Parka -- NOR is expected to stock out within 10 days.
4. [Stockout] Arctic Parka -- ITA is expected to stock out within 4 days.
5. [Stockout] Arctic Parka -- WCA is expected to stock out within 4 days.


## Investigate — the top decision

In [3]:
top = top5.iloc[0]
print(f"Decision: {top.headline}\n")
if top.decision_type in ("Overstock", "Stockout"):
    detail = con.execute(f"""
        SELECT location_id, style_id, on_hand_units, weeks_of_supply, trailing_avg_weekly_sales
        FROM gold.inventory_imbalance_signals
        WHERE region_code = '{top.region_code}' AND style_id = '{top.style_id}' AND location_type='Store'
        ORDER BY on_hand_units DESC LIMIT 8
    """).df()
    detail

Decision: DC-EUCEN delay may expose $3,673,745 of revenue over the next three weeks.



## Simulate — Option A / B / C

In [4]:
# Reuses the same transfer-vs-hold-vs-digital framing as notebook 06.
options = pd.DataFrame([
    {"Option": "A -- Transfer inventory", "Expected incremental revenue": 640000, "Cost": 21000, "Risk": "Medium"},
    {"Option": "B -- Hold inventory", "Expected incremental revenue": -410000, "Cost": 0, "Risk": "High"},
    {"Option": "C -- Increase digital allocation", "Expected incremental revenue": 520000, "Cost": 8000, "Risk": "Low"},
])
fig = px.bar(options, x="Option", y="Expected incremental revenue", color="Option",
             color_discrete_sequence=CATEGORICAL)
fig.update_layout(showlegend=False)
style_fig(fig, "Option comparison for the top decision")
options

,Option,Expected incremental revenue,Cost,Risk
0,A -- Transfer inventory,640000,21000,Medium
1,B -- Hold inventory,-410000,0,High
2,C -- Increase digital allocation,520000,8000,Low


**Approve Option A.** *(See notebook 06 for the fully worked version of this
approval, including the write to `gold.approved_actions`.)*

This is the point of the whole exercise: **the AI is not merely answering a
question — it is helping the business make a better decision**, with a human
firmly in the approval loop.

## Executive rollup

In [5]:
overstock_cost = con.execute("""
    SELECT SUM(GREATEST(overstock_units,0) * current_retail_price * 0.35)
    FROM gold.inventory_imbalance_signals WHERE overstock_pct > 0.2
""").fetchone()[0] or 0
stockout_cost = con.execute("""
    SELECT SUM(sig.trailing_avg_weekly_sales * 3 * sig.current_retail_price)
    FROM gold.inventory_imbalance_signals sig WHERE sig.stockout_est_days < 14
""").fetchone()[0] or 0
supply_risk_cost = con.execute("SELECT SUM(revenue_at_risk) FROM (SELECT trailing_avg_weekly_sales*3*current_retail_price AS revenue_at_risk FROM gold.supply_risk_exposure)").fetchone()[0] or 0
# Scoped to the last 12 weeks, like the other exposure figures above -- summing
# 3 years of historical returns would swamp every other number and isn't what
# "this season" means.
returns_cost = con.execute("""
    SELECT SUM(r.units_returned * 0.5 * sku.current_retail_price)
    FROM silver.fact_returns_line r
    JOIN silver.dim_sku sku ON sku.sku_id = r.sku_id
    WHERE r.week_id >= (SELECT week_id FROM silver.dim_week WHERE is_as_of_week) - 12
""").fetchone()[0] or 0

total_exposure = overstock_cost + stockout_cost + supply_risk_cost + returns_cost
recovery_rate = 0.55  # assumed capture rate on approved actions vs. full exposure
total_recovery = total_exposure * recovery_rate

print(f"Issues identified this season: {len(dq)}")
print(f"  Overstock / markdown exposure : ${overstock_cost:,.0f}")
print(f"  Stockout revenue at risk      : ${stockout_cost:,.0f}")
print(f"  Supply-chain exposure         : ${supply_risk_cost:,.0f}")
print(f"  Returns cost                  : ${returns_cost:,.0f}")
print(f"  ------------------------------------------------")
print(f"  Total exposure identified     : ${total_exposure:,.0f}")
print(f"  Recommended actions recover   : ${total_recovery:,.0f}  (assumed {recovery_rate:.0%} capture)")

Issues identified this season: 299
  Overstock / markdown exposure : $155,840,148
  Stockout revenue at risk      : $79,933,040
  Supply-chain exposure         : $3,673,745
  Returns cost                  : $134,562,743
  ------------------------------------------------
  Total exposure identified     : $374,009,675
  Recommended actions recover   : $205,705,321  (assumed 55% capture)


## North star

**What changed? Why does it matter? What will happen next? What decision should we
make? What are our options? What is the expected impact?** — answered continuously,
with humans responsible for the consequential calls. That's the core of an
AI-native retail decision system.